# 03 — Data Ingestion & Transformation Patterns

Real sources are messy. This notebook covers reading multiple raw formats defensively, then the
transformation patterns (window functions, pivots, UDFs, pandas UDFs, joins) used constantly in
production Fabric pipelines. Scenario: a healthcare patient-visit dataset (patterned after the
MedInsight-style workloads) landing as CSV/JSON from different source systems.


## 1. Reading with explicit schemas and malformed-record handling

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

visit_schema = StructType([
    StructField("patient_id", StringType(), False),
    StructField("visit_date", DateType(), True),
    StructField("department", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("diagnosis_code", StringType(), True),
])

df_visits = (
    spark.read
    .schema(visit_schema)
    .option("header", True)
    .option("mode", "PERMISSIVE")                 # keep bad rows, capture them in _corrupt_record
    .option("columnNameOfCorruptRecord", "_corrupt_record")
    .csv("Files/raw/patient_visits/*.csv")
)

df_bad_rows = df_visits.filter("_corrupt_record IS NOT NULL") if "_corrupt_record" in df_visits.columns else None
df_visits.show(5)


In [ ]:
# Nested / semi-structured JSON from another source system
df_json = spark.read.option("multiLine", True).json("Files/raw/patient_visits_ext/*.json")
df_json.printSchema()

# Flatten a nested struct column
from pyspark.sql import functions as F
df_flat = df_json.select(
    "patient_id",
    F.col("insurance.provider").alias("insurance_provider"),
    F.col("insurance.policy_no").alias("policy_no"),
    F.explode_outer("procedures").alias("procedure"),
)
df_flat.show(10, truncate=False)


## 2. Cleaning: nulls, dedup, type casting

In [ ]:
df_clean = (
    df_visits
    .dropDuplicates(["patient_id", "visit_date", "department"])
    .fillna({"department": "UNKNOWN"})
    .filter(F.col("patient_id").isNotNull())
    .withColumn("age", F.when(F.col("age") < 0, None).otherwise(F.col("age")))
)
df_clean.show(5)


## 3. Window functions

Common in every fact-table pipeline: running totals, ranking, gap analysis.

In [ ]:
from pyspark.sql.window import Window

w = Window.partitionBy("patient_id").orderBy("visit_date")

df_ranked = (
    df_clean
    .withColumn("visit_number", F.row_number().over(w))
    .withColumn("days_since_prev_visit",
                F.datediff("visit_date", F.lag("visit_date").over(w)))
    .withColumn("dept_rank_by_age",
                F.dense_rank().over(Window.partitionBy("department").orderBy(F.desc("age"))))
)
df_ranked.show(10)


## 4. Pivoting for reporting-style outputs

In [ ]:
df_pivot = (
    df_clean.groupBy("department")
    .pivot("diagnosis_code")
    .agg(F.count("*"))
    .fillna(0)
)
df_pivot.show()


## 5. UDFs and pandas UDFs

Prefer built-in functions first — reach for a UDF only when there's no native equivalent.
Pandas UDFs (vectorized) are far faster than row-at-a-time Python UDFs.

In [ ]:
from pyspark.sql.types import StringType
import pyspark.sql.functions as F

# Plain Python UDF (row-at-a-time — slower, avoid on large data if avoidable)
def risk_bucket(age):
    if age is None:
        return "UNKNOWN"
    return "HIGH" if age >= 65 else "MODERATE" if age >= 40 else "LOW"

risk_udf = F.udf(risk_bucket, StringType())
df_clean.withColumn("risk_bucket", risk_udf("age")).show(5)


In [ ]:
from pyspark.sql.functions import pandas_udf
import pandas as pd

# Pandas UDF — operates on a whole Arrow-backed batch at once, much faster
@pandas_udf(StringType())
def risk_bucket_vectorized(age: pd.Series) -> pd.Series:
    return pd.cut(
        age.fillna(-1),
        bins=[-2, -1, 39, 64, 200],
        labels=["UNKNOWN", "LOW", "MODERATE", "HIGH"],
    ).astype(str)

df_clean.withColumn("risk_bucket", risk_bucket_vectorized("age")).show(5)


## 6. Joins: shapes and the broadcast hint

In [ ]:
df_departments = spark.createDataFrame([
    ("CARDIO", "Cardiology", "Bldg A"),
    ("ORTHO", "Orthopedics", "Bldg B"),
    ("UNKNOWN", "Unassigned", "N/A"),
], ["department", "department_name", "building"])

# Small dimension table -> broadcast join avoids a shuffle of the large fact table
df_enriched = df_clean.join(F.broadcast(df_departments), on="department", how="left")
df_enriched.show(5)

# Anti-join: find patients with a visit but no matching insurance record
df_no_insurance = df_clean.join(df_flat, "patient_id", "left_anti")


## 7. Writing the cleaned Silver table

In [ ]:
(
    df_enriched.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_patient_visits")
)


Next notebook: **04 — Spark SQL & notebookutils / mssparkutils**.